# Building LTE and 5G Signal-Quality Labels

This notebook constructs LTE and 5G NR signal-quality labels for 1 km × 1 km grid cells across England.

The workflow:

1. creates a national 1 km grid
2. processes Ofcom LTE and 5G NR measurements
3. assigns measurements to grid cells
4. calculates grid-level RSRP statistics
5. classifies signal quality as Excellent, Good, or Poor and
6. exports the labelled baseline dataset

※ Related dissertation sections:
 - Section 1.2 (Research Aim and Objectives)
 - Section 3.1 (Research Approach)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset19: LTE and 5G NR Label Construction
# Cell 1. Environment Setup
# ============================================================

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Standard libraries
import os

# Data-processing libraries
import numpy as np
import pandas as pd
import geopandas as gpd

# Geometry construction
from shapely.geometry import box

# Display environment information
print("Environment setup complete.")
print(f"NumPy version     : {np.__version__}")
print(f"Pandas version    : {pd.__version__}")
print(f"GeoPandas version : {gpd.__version__}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment setup complete.
NumPy version     : 2.0.2
Pandas version    : 2.2.2
GeoPandas version : 1.1.4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment setup complete.
NumPy version     : 2.0.2
Pandas version    : 2.2.2
GeoPandas version : 1.1.4


## 1. Environment and Output Structure

This section mounts Google Drive, imports the required libraries, creates the project directories, and authenticates Google Earth Engine.

※ Related dissertation sections
 - Section 3.1 (Research Approach—Reproducibility) and Appendix C (Reproducibility and Predictor Definitions).

In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 2. Create Dataset Build Folder Structure
# ============================================================

base_dir = (
    "/content/drive/MyDrive/Dissertation/Experiments/"
    "최종실험(수정)/Dataset_Build"
)

folders = [
    "00_Grid",
    "01_Ofcom",
    "02_Sentinel1/batches",
    "03_Sentinel2/batches",
    "04_WorldCover/batches",
    "05_Population/batches",
    "06_Nightlight/batches",
    "07_OpenCellID",
    "08_Final_Datasets"
]

for folder in folders:
    folder_path = os.path.join(base_dir, folder)
    os.makedirs(folder_path, exist_ok=True)

print("Base directory:")
print(base_dir)

print("\nFolder structure created successfully:")
for folder in folders:
    print(os.path.join(base_dir, folder))

Base directory:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build

Folder structure created successfully:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/00_Grid
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/01_Ofcom
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/02_Sentinel1/batches
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/03_Sentinel2/batches
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/04_WorldCover/batches
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/05_Population/batches
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/06_Nightlight/batches
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/07_OpenCellID
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 3. Authenticate and Initialize Google Earth Engine
# ============================================================

from google.colab import auth
import ee

# Authenticate Google account
auth.authenticate_user()

# Initialize Google Earth Engine
ee.Initialize(project="cw-project-479516")

print("Google authentication successful.")
print("Google Earth Engine initialized successfully.")

Google authentication successful.
Google Earth Engine initialized successfully.


## 2. England Boundary and 1 km Grid

The England boundary is extracted from the GADM 4.1 administrative boundary dataset and reprojected to the British National Grid (EPSG:27700).

A regular 1 km × 1 km spatial grid is then generated across England. Each grid cell receives a unique 'grid_id', which is used to combine signal measurements and geospatial predictors throughout the project.

※ Related dissertation sections
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 4. Download GADM 4.1 Boundary Data
# ============================================================

import requests

# Define source and output paths
gadm_url = (
    "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/"
    "gadm41_GBR.gpkg"
)

gadm_path = os.path.join(
    base_dir,
    "00_Grid",
    "gadm41_GBR.gpkg"
)

# Download only if the file does not already exist
if not os.path.exists(gadm_path):
    response = requests.get(
        gadm_url,
        stream=True,
        timeout=120
    )
    response.raise_for_status()

    with open(gadm_path, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

    print("GADM boundary downloaded successfully.")
else:
    print("GADM boundary already exists.")

# Validate downloaded file
print("\nGADM file:", gadm_path)
print("File exists:", os.path.exists(gadm_path))
print(
    "File size (MB):",
    round(os.path.getsize(gadm_path) / (1024**2), 2)
)

GADM boundary already exists.

GADM file: /content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/00_Grid/gadm41_GBR.gpkg
File exists: True
File size (MB): 411.55


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 5. Read GADM Level-1 Boundary
# ============================================================

# Read GADM Level-1 administrative boundaries
gdf = gpd.read_file(
    gadm_path,
    layer="ADM_ADM_1"
)

# Inspect the dataset
print("GADM Level-1 Boundary")
print("---------------------")
print("Shape:", gdf.shape)
print("CRS:", gdf.crs)

print("\nColumns:")
print(gdf.columns.tolist())

print("\nFirst five records:")
display(gdf.head())

In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 6. Extract and Save England Boundary
# ============================================================

# Extract England using its GADM identifier
england = gdf[
    gdf["GID_1"] == "GBR.1_1"
].copy()

# Validate extraction
if england.empty:
    raise ValueError("England boundary was not found in the GADM dataset.")

print("England boundary records:", len(england))

print("\nSelected administrative unit:")
display(
    england[
        ["GID_1", "COUNTRY", "NAME_1", "ISO_1"]
    ]
)

# Define output path
england_boundary_path = os.path.join(
    base_dir,
    "00_Grid",
    "england_boundary.gpkg"
)

# Save England boundary
england.to_file(
    england_boundary_path,
    layer="england_boundary",
    driver="GPKG"
)

print("\nEngland boundary saved successfully:")
print(england_boundary_path)

England boundary records: 1

Selected administrative unit:


,GID_1,COUNTRY,NAME_1,ISO_1
3,GBR.1_1,United Kingdom,NA,NA



England boundary saved successfully:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/00_Grid/england_boundary.gpkg


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 7. Reproject and Validate England Boundary
# ============================================================

# Reproject to British National Grid
england_bng = england.to_crs(epsg=27700)

# Calculate boundary area
area_km2 = england_bng.geometry.area.sum() / 1_000_000

# Validate the result
print("England Boundary - British National Grid")
print("------------------------------------------")
print("CRS:", england_bng.crs)
print("Bounds [minx, miny, maxx, maxy]:")
print(england_bng.total_bounds)
print("Geometry type:", england_bng.geometry.geom_type.tolist())
print("Area (km²):", round(area_km2, 2))

if not 120_000 <= area_km2 <= 140_000:
    raise ValueError(
        "The boundary area is outside the expected range for England."
    )

print("\nEngland boundary validation successful.")

England Boundary - British National Grid
------------------------------------------
CRS: EPSG:27700
Bounds [minx, miny, maxx, maxy]:
[ 82638.81180731   5401.66355799 655710.82660075 657599.18508027]
Geometry type: ['MultiPolygon']
Area (km²): 130834.11

England boundary validation successful.


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 8. Create England 1 km × 1 km Grid
# ============================================================

import shapely
from rasterio.features import rasterize
from rasterio.transform import from_origin

# Align England extent to 1 km coordinates
minx, miny, maxx, maxy = england_bng.total_bounds

minx = np.floor(minx / 1000) * 1000
miny = np.floor(miny / 1000) * 1000
maxx = np.ceil(maxx / 1000) * 1000
maxy = np.ceil(maxy / 1000) * 1000

ncols = int((maxx - minx) / 1000)
nrows = int((maxy - miny) / 1000)

# Rasterize the England boundary
transform = from_origin(minx, maxy, 1000, 1000)

england_mask = rasterize(
    [(england_bng.geometry.iloc[0], 1)],
    out_shape=(nrows, ncols),
    transform=transform,
    fill=0,
    all_touched=True,
    dtype="uint8"
)

# Obtain selected grid positions
rows, cols = np.where(england_mask == 1)

x_left = minx + cols * 1000
y_top = maxy - rows * 1000

# Create polygons only for selected cells
grid_cells = shapely.box(
    x_left,
    y_top - 1000,
    x_left + 1000,
    y_top
)

england_grid = gpd.GeoDataFrame(
    {"grid_id": np.arange(1, len(grid_cells) + 1)},
    geometry=grid_cells,
    crs="EPSG:27700"
)

print("England grid cells:", f"{len(england_grid):,}")
print("CRS:", england_grid.crs)

display(england_grid.head())

England grid cells: 133,440
CRS: EPSG:27700


,grid_id,geometry
0,1,"POLYGON ((398000 657000, 398000 658000, 397000..."
1,2,"POLYGON ((399000 657000, 399000 658000, 398000..."
2,3,"POLYGON ((396000 656000, 396000 657000, 395000..."
3,4,"POLYGON ((397000 656000, 397000 657000, 396000..."
4,5,"POLYGON ((398000 656000, 398000 657000, 397000..."


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 9. Validate and Save England 1 km Grid
# ============================================================

grid_england = england_grid.copy()

# Validate grid
print("England 1 km Grid Validation")
print("-----------------------------")
print("Number of grid cells:", f"{len(grid_england):,}")
print("CRS:", grid_england.crs)
print("Unique grid IDs:", grid_england["grid_id"].nunique())
print("Duplicate grid IDs:", grid_england["grid_id"].duplicated().sum())
print("Missing geometries:", grid_england.geometry.isna().sum())
print("Invalid geometries:", (~grid_england.geometry.is_valid).sum())

# Stop if validation fails
if grid_england["grid_id"].duplicated().any():
    raise ValueError("Duplicate grid IDs were found.")

if grid_england.geometry.isna().any():
    raise ValueError("Missing grid geometries were found.")

if (~grid_england.geometry.is_valid).any():
    raise ValueError("Invalid grid geometries were found.")

# Define output path
grid_path = os.path.join(
    base_dir,
    "00_Grid",
    "england_1km_grid.gpkg"
)

# Save grid
grid_england.to_file(
    grid_path,
    layer="england_1km_grid",
    driver="GPKG"
)

print("\nGrid saved successfully:")
print(grid_path)

England 1 km Grid Validation
-----------------------------
Number of grid cells: 133,440
CRS: EPSG:27700
Unique grid IDs: 133440
Duplicate grid IDs: 0
Missing geometries: 0
Invalid geometries: 0

Grid saved successfully:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/00_Grid/england_1km_grid.gpkg


## 3. Ofcom Signal-Measurement Data

The signal labels are derived from the 2025 Ofcom mobile signal measurement datasets:

- 4G LTE measurement data
- 5G NR measurement data

Only the coordinates, observation period, and required RSRP fields are loaded. Large CSV files are processed in chunks to reduce memory usage.

※Related dissertation sections
 - Section 3.1 (Research Approach—Data Understanding and Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 10. Define and Validate Ofcom Source Files
# ============================================================

import os

ofcom_dir = "/content/drive/MyDrive/Dissertation/Ofcom dataset"

lte_file = os.path.join(
    ofcom_dir,
    "4g-lte-2025-mobile-signal-measurement-data.csv"
)

nr_file = os.path.join(
    ofcom_dir,
    "5g-nr-2025-mobile-signal-measurement-data.csv"
)

print("Ofcom Source File Validation")
print("----------------------------")

print("LTE file exists:", os.path.exists(lte_file))
print("NR file exists :", os.path.exists(nr_file))

print("\nLTE source:")
print(lte_file)

print("\nNR source:")
print(nr_file)

if os.path.exists(lte_file):
    print(
        "\nLTE file size:",
        round(os.path.getsize(lte_file) / (1024**3), 2),
        "GB"
    )

if os.path.exists(nr_file):
    print(
        "NR file size :",
        round(os.path.getsize(nr_file) / (1024**3), 2),
        "GB"
    )

Ofcom Source File Validation
----------------------------
LTE file exists: True
NR file exists : True

LTE source:
/content/drive/MyDrive/Dissertation/Ofcom dataset/4g-lte-2025-mobile-signal-measurement-data.csv

NR source:
/content/drive/MyDrive/Dissertation/Ofcom dataset/5g-nr-2025-mobile-signal-measurement-data.csv

LTE file size: 7.03 GB
NR file size : 3.29 GB


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 11. Define Required Ofcom Columns
# ============================================================

# Common columns
common_cols = [
    "latitude",
    "longitude",
    "month_year"
]

# LTE RSRP columns
lte_rsrp_cols = [
    "rsrp_top1_3uk", "rsrp_top2_3uk", "rsrp_top3_3uk", "rsrp_top4_3uk",
    "rsrp_top1_ee",  "rsrp_top2_ee",  "rsrp_top3_ee",  "rsrp_top4_ee",
    "rsrp_top1_o2",  "rsrp_top2_o2",  "rsrp_top3_o2",  "rsrp_top4_o2",
    "rsrp_top1_vf",  "rsrp_top2_vf",  "rsrp_top3_vf",  "rsrp_top4_vf"
]

# NR RSRP columns
nr_rsrp_cols = [
    "rsrp_top1_3uk",
    "rsrp_top1_ee",
    "rsrp_top1_o2",
    "rsrp_top1_vf"
]

lte_usecols = common_cols + lte_rsrp_cols
nr_usecols = common_cols + nr_rsrp_cols

print("LTE columns to load:", len(lte_usecols))
print("NR columns to load :", len(nr_usecols))

print("\nLTE RSRP columns:", len(lte_rsrp_cols))
print("NR RSRP columns :", len(nr_rsrp_cols))

LTE columns to load: 19
NR columns to load : 7

LTE RSRP columns: 16
NR RSRP columns : 4


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 12. Check Ofcom Month Distribution
# ============================================================

from collections import Counter

chunk_size = 500_000

lte_month_counts = Counter()
nr_month_counts = Counter()

# Count LTE records by month
for chunk in pd.read_csv(
    lte_file,
    usecols=["month_year"],
    chunksize=chunk_size
):
    lte_month_counts.update(
        chunk["month_year"].value_counts().to_dict()
    )

# Count 5G NR records by month
for chunk in pd.read_csv(
    nr_file,
    usecols=["month_year"],
    chunksize=chunk_size
):
    nr_month_counts.update(
        chunk["month_year"].value_counts().to_dict()
    )

print("LTE month distribution:")
print(pd.Series(lte_month_counts).sort_index())

print("\n5G NR month distribution:")
print(pd.Series(nr_month_counts).sort_index())

LTE month distribution:
2025-08-01      367827
2025-09-01    11536585
2025-10-01      162500
dtype: int64

5G NR month distribution:
2025-08-01      809747
2025-09-01    24716047
2025-10-01      355495
dtype: int64


## 4. LTE Label Construction

For each LTE observation, the strongest available RSRP value is selected across the included operator and ranked-signal fields.

Valid observations are converted to spatial points and assigned to the England 1 km grid. The grid-level minimum, mean, median, and observation count are then calculated.

Signal-quality classes are defined using the median LTE RSRP:

- Excellent: RSRP > −80 dBm
- Good: −90 dBm < RSRP ≤ −80 dBm
- Poor: RSRP ≤ −90 dBm

※ Related dissertation sections
 - Section 2.3 (Mobile Signal-Quality Indicators)
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 13. Prepare Raw LTE Processing
# ============================================================

# Reuse the validated LTE source path
lte_raw_path = lte_file

# Define output directory
lte_output_dir = os.path.join(
    base_dir,
    "01_Ofcom",
    "LTE_grid_chunks"
)

os.makedirs(lte_output_dir, exist_ok=True)

# Validate paths
if not os.path.exists(lte_raw_path):
    raise FileNotFoundError(f"LTE source file not found: {lte_raw_path}")

print("LTE raw file:", lte_raw_path)
print("LTE raw exists:", os.path.exists(lte_raw_path))
print("Output directory:", lte_output_dir)

LTE raw file: /content/drive/MyDrive/Dissertation/Ofcom dataset/4g-lte-2025-mobile-signal-measurement-data.csv
LTE raw exists: True
Output directory: /content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/01_Ofcom/LTE_grid_chunks


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 14. Check Raw LTE Columns
# ============================================================

lte_sample = pd.read_csv(
    lte_raw_path,
    nrows=5,
    low_memory=False
)

print("Sample shape:", lte_sample.shape)

print("\nColumns:")
print(lte_sample.columns.tolist())

print("\nFirst five rows:")
display(lte_sample.head())

Sample shape: (5, 165)

Columns:
['latitude', 'longitude', 'month_year', 'hour_ref', 'rnum', 'pci_top1_3uk', 'power_top1_3uk', 'sinr_top1_3uk', 'rsrp_top1_3uk', 'rsrq_top1_3uk', 'earfcn_top1_3uk', 'add_plmn_top1_3uk', 'mcc_top1_3uk', 'mnc_top1_3uk', 'nr_top1_3uk', 'pci_top2_3uk', 'power_top2_3uk', 'sinr_top2_3uk', 'rsrp_top2_3uk', 'rsrq_top2_3uk', 'earfcn_top2_3uk', 'add_plmn_top2_3uk', 'mcc_top2_3uk', 'mnc_top2_3uk', 'nr_top2_3uk', 'pci_top3_3uk', 'power_top3_3uk', 'sinr_top3_3uk', 'rsrp_top3_3uk', 'rsrq_top3_3uk', 'earfcn_top3_3uk', 'add_plmn_top3_3uk', 'mcc_top3_3uk', 'mnc_top3_3uk', 'nr_top3_3uk', 'pci_top4_3uk', 'power_top4_3uk', 'sinr_top4_3uk', 'rsrp_top4_3uk', 'rsrq_top4_3uk', 'earfcn_top4_3uk', 'add_plmn_top4_3uk', 'mcc_top4_3uk', 'mnc_top4_3uk', 'nr_top4_3uk', 'pci_top1_ee', 'power_top1_ee', 'sinr_top1_ee', 'rsrp_top1_ee', 'rsrq_top1_ee', 'earfcn_top1_ee', 'add_plmn_top1_ee', 'mcc_top1_ee', 'mnc_top1_ee', 'nr_top1_ee', 'pci_top2_ee', 'power_top2_ee', 'sinr_top2_ee', 'rsrp_top

,latitude,longitude,month_year,hour_ref,rnum,pci_top1_3uk,power_top1_3uk,sinr_top1_3uk,rsrp_top1_3uk,rsrq_top1_3uk,...,pci_top4_vf,power_top4_vf,sinr_top4_vf,rsrp_top4_vf,rsrq_top4_vf,earfcn_top4_vf,add_plmn_top4_vf,mcc_top4_vf,mnc_top4_vf,nr_top4_vf
0,53.398871,-2.510949,2025-08-01,88753482,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,53.398901,-2.510974,2025-08-01,88753482,2,181.0,-58.03,6.72,-80.32,-16.22,...,317.0,-47.38,24.89,-70.87,-15.14,323.0,NaN,234.0,15.0,yes
2,53.398936,-2.511002,2025-08-01,88753482,3,181.0,-58.03,6.72,-80.32,-16.22,...,317.0,-47.38,24.89,-70.87,-15.14,323.0,NaN,234.0,15.0,yes
3,53.398959,-2.511021,2025-08-01,88753482,4,181.0,-58.03,6.72,-80.32,-16.22,...,317.0,-47.38,24.89,-70.87,-15.14,323.0,NaN,234.0,15.0,yes
4,53.398992,-2.511048,2025-08-01,88753482,5,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 15. Test LTE Raw Chunk Processing
# ============================================================

test_raw = pd.read_csv(
    lte_raw_path,
    usecols=["latitude", "longitude"] + lte_rsrp_cols,
    nrows=500_000
)

# Select the strongest available LTE RSRP
test_raw["lte_best_rsrp"] = test_raw[
    lte_rsrp_cols
].max(axis=1)

# Remove missing and invalid coordinates
test_raw = test_raw.dropna(
    subset=["latitude", "longitude", "lte_best_rsrp"]
)

test_raw = test_raw[
    test_raw["latitude"].between(49, 56)
    & test_raw["longitude"].between(-7, 3)
].copy()

print("Valid LTE records:", f"{len(test_raw):,}")

display(
    test_raw[
        ["latitude", "longitude", "lte_best_rsrp"]
    ].head()
)

Valid LTE records: 301,013


,latitude,longitude,lte_best_rsrp
1,53.398901,-2.510974,-52.21
2,53.398936,-2.511002,-52.21
3,53.398959,-2.511021,-52.21
5,53.399003,-2.511056,-52.21
6,53.399023,-2.511073,-52.21


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 16. Test LTE Spatial Join to 1 km Grid
# ============================================================

test_lte_points = gpd.GeoDataFrame(
    test_raw[["lte_best_rsrp"]],
    geometry=gpd.points_from_xy(
        test_raw["longitude"],
        test_raw["latitude"]
    ),
    crs="EPSG:4326"
).to_crs(epsg=27700)

test_lte_joined = gpd.sjoin(
    test_lte_points,
    grid_england[["grid_id", "geometry"]],
    how="inner",
    predicate="within"
)

print("Input valid LTE points:", len(test_lte_points))
print("Joined LTE points:", len(test_lte_joined))

display(test_lte_joined.head())

Input valid LTE points: 301013
Joined LTE points: 263031


,lte_best_rsrp,geometry,index_right,grid_id
1,-52.21,POINT (366121.786 389241.052),37176,37177
2,-52.21,POINT (366119.896 389245.022),37176,37177
3,-52.21,POINT (366118.695 389247.547),37176,37177
5,-52.21,POINT (366116.355 389252.444),37176,37177
6,-52.21,POINT (366115.272 389254.69),37176,37177


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 17. Process Full Raw LTE CSV into 1 km Grid Chunks
# ============================================================

chunksize = 500_000

for i, chunk in enumerate(
    pd.read_csv(
        lte_raw_path,
        usecols=["latitude", "longitude"] + lte_rsrp_cols,
        chunksize=chunksize
    )
):
    print(f"Processing LTE chunk {i:03d}")

    # Select strongest available LTE RSRP
    chunk["lte_best_rsrp"] = chunk[
        lte_rsrp_cols
    ].max(axis=1)

    # Remove missing and invalid records
    chunk = chunk.dropna(
        subset=["latitude", "longitude", "lte_best_rsrp"]
    )

    chunk = chunk[
        chunk["latitude"].between(49, 56)
        & chunk["longitude"].between(-7, 3)
    ].copy()

    # Convert measurement points to British National Grid
    points = gpd.GeoDataFrame(
        chunk[["lte_best_rsrp"]],
        geometry=gpd.points_from_xy(
            chunk["longitude"],
            chunk["latitude"]
        ),
        crs="EPSG:4326"
    ).to_crs(epsg=27700)

    # Assign each measurement to a 1 km grid cell
    joined = gpd.sjoin(
        points,
        grid_england[["grid_id", "geometry"]],
        how="inner",
        predicate="within"
    )

    # Retain raw grid-level measurements for exact aggregation
    chunk_output = joined[
        ["grid_id", "lte_best_rsrp"]
    ].copy()

    out_path = os.path.join(
        lte_output_dir,
        f"dataset19_lte_joined_chunk_{i:03d}.csv"
    )

    chunk_output.to_csv(out_path, index=False)

    print(
        f"Saved chunk {i:03d} | "
        f"joined records: {len(chunk_output):,}"
    )

print("All LTE chunks processed successfully.")

Processing LTE chunk 000
Saved chunk 000 | joined records: 263,031
Processing LTE chunk 001
Saved chunk 001 | joined records: 191,774
Processing LTE chunk 002
Saved chunk 002 | joined records: 198,702
Processing LTE chunk 003
Saved chunk 003 | joined records: 204,874
Processing LTE chunk 004
Saved chunk 004 | joined records: 207,262
Processing LTE chunk 005
Saved chunk 005 | joined records: 261,156
Processing LTE chunk 006
Saved chunk 006 | joined records: 243,675
Processing LTE chunk 007
Saved chunk 007 | joined records: 144,097
Processing LTE chunk 008
Saved chunk 008 | joined records: 216,241
Processing LTE chunk 009
Saved chunk 009 | joined records: 205,534
Processing LTE chunk 010
Saved chunk 010 | joined records: 213,176
Processing LTE chunk 011
Saved chunk 011 | joined records: 254,801
Processing LTE chunk 012
Saved chunk 012 | joined records: 229,463
Processing LTE chunk 013
Saved chunk 013 | joined records: 201,347
Processing LTE chunk 014
Saved chunk 014 | joined records: 191

In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 18. Merge and Aggregate LTE Measurement Chunks
# ============================================================

# Identify newly created joined LTE chunks
lte_joined_chunk_files = sorted([
    os.path.join(lte_output_dir, filename)
    for filename in os.listdir(lte_output_dir)
    if filename.startswith("dataset19_lte_joined_chunk_")
    and filename.endswith(".csv")
])

print("Number of LTE joined chunk files:", len(lte_joined_chunk_files))

if len(lte_joined_chunk_files) != 25:
    raise ValueError(
        f"Expected 25 LTE chunks, but found {len(lte_joined_chunk_files)}."
    )

# Load and combine all grid-level LTE measurements
lte_all = pd.concat(
    [
        pd.read_csv(
            file,
            dtype={
                "grid_id": "int32",
                "lte_best_rsrp": "float32"
            }
        )
        for file in lte_joined_chunk_files
    ],
    ignore_index=True
)

print("Total joined LTE records:", f"{len(lte_all):,}")

# Calculate exact grid-level statistics
lte_labels = (
    lte_all
    .groupby("grid_id", as_index=False)
    .agg(
        lte_min_rsrp=("lte_best_rsrp", "min"),
        lte_mean_rsrp=("lte_best_rsrp", "mean"),
        lte_median_rsrp=("lte_best_rsrp", "median"),
        lte_point_count=("lte_best_rsrp", "count")
    )
)

print("LTE labelled grid cells:", f"{len(lte_labels):,}")

display(lte_labels.head())

Number of LTE joined chunk files: 25
Total joined LTE records: 5,216,913
LTE labelled grid cells: 4,970


,grid_id,lte_min_rsrp,lte_mean_rsrp,lte_median_rsrp,lte_point_count
0,3975,-55.639999,-46.228794,-45.480000,937
1,3976,-81.709999,-68.405396,-69.129997,1609
2,4072,-89.989998,-83.527565,-84.709999,1134
3,4166,-74.809998,-64.096870,-62.380001,624
4,4167,-74.290001,-70.279633,-70.440002,216


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 19. Create and Save Final LTE Signal Labels
# ============================================================

# Create LTE classes from grid-level median RSRP
lte_labels["lte_signal_class"] = np.select(
    [
        lte_labels["lte_median_rsrp"] > -80,
        lte_labels["lte_median_rsrp"] > -90
    ],
    [
        "Excellent",
        "Good"
    ],
    default="Poor"
)

# Validate classes
print("LTE class distribution:")
print(lte_labels["lte_signal_class"].value_counts())

print("\nMissing LTE classes:")
print(lte_labels["lte_signal_class"].isna().sum())

# Define output path
lte_final_path = os.path.join(
    base_dir,
    "01_Ofcom",
    "dataset19_lte_labels_final.csv"
)

# Save final LTE labels
lte_labels.to_csv(
    lte_final_path,
    index=False
)

print("\nFinal LTE labelled grids:", f"{len(lte_labels):,}")
print("Saved:")
print(lte_final_path)

LTE class distribution:
lte_signal_class
Excellent    4077
Good          772
Poor          121
Name: count, dtype: int64

Missing LTE classes:
0

Final LTE labelled grids: 4,970
Saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/01_Ofcom/dataset19_lte_labels_final.csv


## 5. 5G NR Label Construction

For each 5G NR observation, the strongest available RSRP value across the included operators is selected.

Valid observations are spatially assigned to the same 1 km grid. The grid-level minimum, mean, median, and observation count are subsequently calculated.

Signal-quality classes are defined using the median 5G NR RSRP:

- Excellent: RSRP > −90 dBm
- Good: −100 dBm < RSRP ≤ −90 dBm
- Poor: RSRP ≤ −100 dBm

※ Related dissertation sections
 - Section 2.3 (Mobile Signal-Quality Indicators)
 - Section 3.1 (Research Approach—Data Preparation)
 - Section 4.1.1 (Baseline Dataset Construction).

In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 20. Prepare and Inspect Raw 5G NR Data
# ============================================================

# Reuse the validated 5G NR source path
nr_raw_path = nr_file

# Define output directory
nr_output_dir = os.path.join(
    base_dir,
    "01_Ofcom",
    "NR_grid_chunks"
)

os.makedirs(nr_output_dir, exist_ok=True)

# Validate source file
if not os.path.exists(nr_raw_path):
    raise FileNotFoundError(f"5G NR source file not found: {nr_raw_path}")

print("5G NR raw file:", nr_raw_path)
print("5G NR raw exists:", os.path.exists(nr_raw_path))
print("Output directory:", nr_output_dir)

# Inspect raw data structure
nr_sample = pd.read_csv(
    nr_raw_path,
    nrows=5,
    low_memory=False
)

print("\nSample shape:", nr_sample.shape)

print("\nColumns:")
print(nr_sample.columns.tolist())

print("\nFirst five rows:")
display(nr_sample.head())

5G NR raw file: /content/drive/MyDrive/Dissertation/Ofcom dataset/5g-nr-2025-mobile-signal-measurement-data.csv
5G NR raw exists: True
Output directory: /content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/01_Ofcom/NR_grid_chunks

Sample shape: (5, 49)

Columns:
['latitude', 'longitude', 'month_year', 'hour_ref', 'rnum', 'pci_top1_3uk', 'rssi_top1_3uk', 'ssb_idx_top1_3uk', 'sinr_top1_3uk', 'rsrp_top1_3uk', 'rsrq_top1_3uk', 'nr_arfcn_top1_3uk', 'add_plmn_top1_3uk', 'mcc_top1_3uk', 'mnc_top1_3uk', 'nr_mode_top1_3uk', 'pci_top1_ee', 'rssi_top1_ee', 'ssb_idx_top1_ee', 'sinr_top1_ee', 'rsrp_top1_ee', 'rsrq_top1_ee', 'nr_arfcn_top1_ee', 'add_plmn_top1_ee', 'mcc_top1_ee', 'mnc_top1_ee', 'nr_mode_top1_ee', 'pci_top1_o2', 'rssi_top1_o2', 'ssb_idx_top1_o2', 'sinr_top1_o2', 'rsrp_top1_o2', 'rsrq_top1_o2', 'nr_arfcn_top1_o2', 'add_plmn_top1_o2', 'mcc_top1_o2', 'mnc_top1_o2', 'nr_mode_top1_o2', 'pci_top1_vf', 'rssi_top1_vf', 'ssb_idx_top1_vf', 'sinr_top1_vf', 'rsrp_top1_vf', 'rsrq

,latitude,longitude,month_year,hour_ref,rnum,pci_top1_3uk,rssi_top1_3uk,ssb_idx_top1_3uk,sinr_top1_3uk,rsrp_top1_3uk,...,rssi_top1_vf,ssb_idx_top1_vf,sinr_top1_vf,rsrp_top1_vf,rsrq_top1_vf,nr_arfcn_top1_vf,add_plmn_top1_vf,mcc_top1_vf,mnc_top1_vf,nr_mode_top1_vf
0,53.401023,-2.513024,2025-08-01,88753482,1,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,53.401041,-2.513045,2025-08-01,88753482,2,766.0,-82.96,2.0,NaN,NaN,...,-48.21,2.0,NaN,NaN,NaN,428190.0,NaN,234.0,15.0,SA
2,53.401075,-2.513086,2025-08-01,88753482,3,766.0,-82.96,2.0,NaN,NaN,...,-26.45,2.0,20.39,-48.14,-10.28,428190.0,NaN,234.0,15.0,SA
3,53.401076,-2.513087,2025-08-01,88753482,4,766.0,-65.68,2.0,NaN,NaN,...,-26.45,2.0,NaN,NaN,NaN,428190.0,NaN,234.0,15.0,SA
4,53.401080,-2.513092,2025-08-01,88753482,5,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,20.77,-49.56,-10.28,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 21. Test 5G NR Raw Chunk Processing
# ============================================================

test_nr = pd.read_csv(
    nr_raw_path,
    usecols=["latitude", "longitude"] + nr_rsrp_cols,
    nrows=500_000
)

# Select the strongest available RSRP across operators
test_nr["nr_best_rsrp"] = test_nr[
    nr_rsrp_cols
].max(axis=1)

# Remove missing records
test_nr = test_nr.dropna(
    subset=["latitude", "longitude", "nr_best_rsrp"]
)

# Retain valid coordinates around England
test_nr = test_nr[
    test_nr["latitude"].between(49, 56)
    & test_nr["longitude"].between(-7, 3)
].copy()

print("Valid 5G NR records:", f"{len(test_nr):,}")

display(
    test_nr[
        ["latitude", "longitude", "nr_best_rsrp"]
    ].head()
)

Valid 5G NR records: 295,743


,latitude,longitude,nr_best_rsrp
1,53.401041,-2.513045,-65.58
2,53.401075,-2.513086,-48.14
4,53.401080,-2.513092,-49.56
5,53.401081,-2.513092,-96.82
6,53.401085,-2.513098,-48.22


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 22. Test NR Spatial Join to 1 km Grid
# ============================================================

test_nr_points = gpd.GeoDataFrame(
    test_nr[["nr_best_rsrp"]],
    geometry=gpd.points_from_xy(
        test_nr["longitude"],
        test_nr["latitude"]
    ),
    crs="EPSG:4326"
).to_crs(epsg=27700)

test_nr_joined = gpd.sjoin(
    test_nr_points,
    grid_england[["grid_id", "geometry"]],
    how="inner",
    predicate="within"
)

print("Input valid NR points:", len(test_nr_points))
print("Joined NR points:", len(test_nr_joined))

display(test_nr_joined.head())

Input valid NR points: 295743
Joined NR points: 280339


,nr_best_rsrp,geometry,index_right,grid_id
1,-65.58,POINT (365985.758 389480.122),37175,37176
2,-48.14,POINT (365983.062 389483.995),37175,37176
4,-49.56,POINT (365982.679 389484.546),37175,37176
5,-96.82,POINT (365982.641 389484.6),37175,37176
6,-48.22,POINT (365982.297 389485.095),37175,37176


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 23. Process Full Raw 5G NR CSV into 1 km Grid Chunks
# ============================================================

chunksize = 500_000

for i, chunk in enumerate(
    pd.read_csv(
        nr_raw_path,
        usecols=["latitude", "longitude"] + nr_rsrp_cols,
        chunksize=chunksize
    )
):
    print(f"Processing 5G NR chunk {i:03d}")

    # Select strongest available 5G NR RSRP
    chunk["nr_best_rsrp"] = chunk[
        nr_rsrp_cols
    ].max(axis=1)

    # Remove missing records
    chunk = chunk.dropna(
        subset=["latitude", "longitude", "nr_best_rsrp"]
    )

    # Retain valid coordinates around England
    chunk = chunk[
        chunk["latitude"].between(49, 56)
        & chunk["longitude"].between(-7, 3)
    ].copy()

    # Convert points to British National Grid
    points = gpd.GeoDataFrame(
        chunk[["nr_best_rsrp"]],
        geometry=gpd.points_from_xy(
            chunk["longitude"],
            chunk["latitude"]
        ),
        crs="EPSG:4326"
    ).to_crs(epsg=27700)

    # Assign each measurement to a 1 km grid cell
    joined = gpd.sjoin(
        points,
        grid_england[["grid_id", "geometry"]],
        how="inner",
        predicate="within"
    )

    # Retain raw measurements for exact final aggregation
    chunk_output = joined[
        ["grid_id", "nr_best_rsrp"]
    ].copy()

    out_path = os.path.join(
        nr_output_dir,
        f"dataset19_nr_joined_chunk_{i:03d}.csv"
    )

    chunk_output.to_csv(out_path, index=False)

    print(
        f"Saved chunk {i:03d} | "
        f"joined records: {len(chunk_output):,}"
    )

print("All 5G NR chunks processed successfully.")

Processing 5G NR chunk 000
Saved chunk 000 | joined records: 280,339
Processing 5G NR chunk 001
Saved chunk 001 | joined records: 185,719
Processing 5G NR chunk 002
Saved chunk 002 | joined records: 128,329
Processing 5G NR chunk 003
Saved chunk 003 | joined records: 164,813
Processing 5G NR chunk 004
Saved chunk 004 | joined records: 182,588
Processing 5G NR chunk 005
Saved chunk 005 | joined records: 185,224
Processing 5G NR chunk 006
Saved chunk 006 | joined records: 164,097
Processing 5G NR chunk 007
Saved chunk 007 | joined records: 147,621
Processing 5G NR chunk 008
Saved chunk 008 | joined records: 181,316
Processing 5G NR chunk 009
Saved chunk 009 | joined records: 144,021
Processing 5G NR chunk 010
Saved chunk 010 | joined records: 203,717
Processing 5G NR chunk 011
Saved chunk 011 | joined records: 228,158
Processing 5G NR chunk 012
Saved chunk 012 | joined records: 205,657
Processing 5G NR chunk 013
Saved chunk 013 | joined records: 206,480
Processing 5G NR chunk 014
Saved c

In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 24. Merge and Aggregate 5G NR Measurement Chunks
# ============================================================

# Identify newly created joined 5G NR chunks
nr_joined_chunk_files = sorted([
    os.path.join(nr_output_dir, filename)
    for filename in os.listdir(nr_output_dir)
    if filename.startswith("dataset19_nr_joined_chunk_")
    and filename.endswith(".csv")
])

print("Number of 5G NR joined chunk files:", len(nr_joined_chunk_files))

if len(nr_joined_chunk_files) != 52:
    raise ValueError(
        f"Expected 52 NR chunks, but found {len(nr_joined_chunk_files)}."
    )

# Load and combine all grid-level measurements
nr_all = pd.concat(
    [
        pd.read_csv(
            file,
            dtype={
                "grid_id": "int32",
                "nr_best_rsrp": "float32"
            }
        )
        for file in nr_joined_chunk_files
    ],
    ignore_index=True
)

print("Total joined 5G NR records:", f"{len(nr_all):,}")

# Calculate exact grid-level statistics
nr_labels = (
    nr_all
    .groupby("grid_id", as_index=False)
    .agg(
        nr_min_rsrp=("nr_best_rsrp", "min"),
        nr_mean_rsrp=("nr_best_rsrp", "mean"),
        nr_median_rsrp=("nr_best_rsrp", "median"),
        nr_point_count=("nr_best_rsrp", "count")
    )
)

print("5G NR labelled grid cells:", f"{len(nr_labels):,}")

display(nr_labels.head())

Number of 5G NR joined chunk files: 52
Total joined 5G NR records: 9,196,375
5G NR labelled grid cells: 4,965


,grid_id,nr_min_rsrp,nr_mean_rsrp,nr_median_rsrp,nr_point_count
0,3975,-128.639999,-89.648033,-92.370003,1755
1,3976,-130.869995,-92.905037,-92.430000,2657
2,4072,-138.300003,-100.509041,-96.830002,2055
3,4166,-128.369995,-90.326965,-93.480003,1207
4,4167,-131.270004,-91.655655,-94.239998,361


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 25. Create 5G NR Signal-Quality Classes
# ============================================================

# Create classes from grid-level median RSRP
nr_labels["nr_signal_class"] = np.select(
    [
        nr_labels["nr_median_rsrp"] > -90,
        nr_labels["nr_median_rsrp"] > -100
    ],
    [
        "Excellent",
        "Good"
    ],
    default="Poor"
)

# Validate classes
print("5G NR class distribution:")
print(nr_labels["nr_signal_class"].value_counts())

print("\nMissing 5G NR classes:")
print(nr_labels["nr_signal_class"].isna().sum())

5G NR class distribution:
nr_signal_class
Excellent    2441
Good         1718
Poor          806
Name: count, dtype: int64

Missing 5G NR classes:
0


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 26. Save Final 5G NR Labels
# ============================================================

# Define output path
nr_final_path = os.path.join(
    base_dir,
    "01_Ofcom",
    "dataset19_nr_labels_final.csv"
)

# Save final 5G NR labels
nr_labels.to_csv(
    nr_final_path,
    index=False
)

print("Final 5G NR class distribution:")
print(nr_labels["nr_signal_class"].value_counts())

print("\nFinal 5G NR labelled grids:", f"{len(nr_labels):,}")

print("\nSaved:")
print(nr_final_path)

display(nr_labels.head())

Final 5G NR class distribution:
nr_signal_class
Excellent    2441
Good         1718
Poor          806
Name: count, dtype: int64

Final 5G NR labelled grids: 4,965

Saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/01_Ofcom/dataset19_nr_labels_final.csv


,grid_id,nr_min_rsrp,nr_mean_rsrp,nr_median_rsrp,nr_point_count,nr_signal_class
0,3975,-128.639999,-89.648033,-92.370003,1755,Good
1,3976,-130.869995,-92.905037,-92.430000,2657,Good
2,4072,-138.300003,-100.509041,-96.830002,2055,Good
3,4166,-128.369995,-90.326965,-93.480003,1207,Good
4,4167,-131.270004,-91.655655,-94.239998,361,Good


## 6. Merge and Export the Baseline Dataset

The LTE and 5G NR statistics and class labels are merged with the complete England grid using `grid_id`.

The final baseline dataset is exported in two formats:

- CSV: attribute data without spatial geometry
- GeoPackage: spatial data including grid geometry

※ Related dissertation sections
 - Section 4.1.1 (Baseline Dataset Construction)
 - Appendix C (Reproducibility and Predictor Definitions).

In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 27. Merge Grid with LTE and 5G NR Labels
# ============================================================

# Start from the complete England 1 km grid
dataset19_base = grid_england[
    ["grid_id", "geometry"]
].copy()

# Merge LTE labels
dataset19_base = dataset19_base.merge(
    lte_labels,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Merge 5G NR labels
dataset19_base = dataset19_base.merge(
    nr_labels,
    on="grid_id",
    how="left",
    validate="one_to_one"
)

# Validate merged dataset
if len(dataset19_base) != len(grid_england):
    raise ValueError("The number of grid cells changed during label merging.")

print("Dataset19 base shape:", dataset19_base.shape)
print(
    "LTE labelled grids:",
    f"{dataset19_base['lte_signal_class'].notna().sum():,}"
)
print(
    "5G NR labelled grids:",
    f"{dataset19_base['nr_signal_class'].notna().sum():,}"
)

display(dataset19_base.head())

Dataset19 base shape: (133440, 12)
LTE labelled grids: 4,970
5G NR labelled grids: 4,965


,grid_id,geometry,lte_min_rsrp,lte_mean_rsrp,lte_median_rsrp,lte_point_count,lte_signal_class,nr_min_rsrp,nr_mean_rsrp,nr_median_rsrp,nr_point_count,nr_signal_class
0,1,"POLYGON ((398000 657000, 398000 658000, 397000...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,"POLYGON ((399000 657000, 399000 658000, 398000...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,"POLYGON ((396000 656000, 396000 657000, 395000...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,"POLYGON ((397000 656000, 397000 657000, 396000...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,"POLYGON ((398000 656000, 398000 657000, 397000...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ============================================================
# Dataset19 (Baseline)
# Cell 28. Save Grid, LTE, and 5G NR Base Dataset
# ============================================================

output_dir = os.path.join(
    base_dir,
    "08_Final_Datasets"
)

# Define output paths
base_csv_path = os.path.join(
    output_dir,
    "dataset19_grid_lte_nr_base.csv"
)

base_gpkg_path = os.path.join(
    output_dir,
    "dataset19_grid_lte_nr_base.gpkg"
)

# Save attribute table without geometry
dataset19_base.drop(
    columns="geometry"
).to_csv(
    base_csv_path,
    index=False
)

# Save spatial dataset with geometry
dataset19_base.to_file(
    base_gpkg_path,
    layer="dataset19_grid_lte_nr_base",
    driver="GPKG"
)

print("Dataset shape:", dataset19_base.shape)

print("\nCSV saved:")
print(base_csv_path)

print("\nGeoPackage saved:")
print(base_gpkg_path)

Dataset shape: (133440, 12)

CSV saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets/dataset19_grid_lte_nr_base.csv

GeoPackage saved:
/content/drive/MyDrive/Dissertation/Experiments/최종실험(수정)/Dataset_Build/08_Final_Datasets/dataset19_grid_lte_nr_base.gpkg
